# RAG Document Assistant — Day 5: Conversational Memory
### Turning isolated one-off questions into a real back-and-forth chat.

**Stack:** Google Gemini 2.5 Flash + Gemini Embeddings + LangChain + ChromaDB

**Recap of Day 4:** we added score-threshold filtering (weak matches get discarded before reaching the model) and citation-labeled context, so answers now cite `[Source N]` and cleanly decline when nothing relevant is found. But every `ask_v2()` call was still fully independent — a follow-up like "tell me more about that" made no sense, because the model had no idea what "that" referred to.

**Today's goal:** fix the "goldfish memory" problem. By the end of today, you can ask a question, then ask a natural follow-up that depends on the previous answer, and the assistant will understand what you mean.

---

## Step 1 — Why This Is Trickier Than It Sounds

You might think "just keep a list of past messages and send them all every time" — and that's *half* the solution, but there's a subtle bug it doesn't fix on its own.

**The problem:** retrieval happens *before* the model ever sees anything. If you ask "What is LangChain?" and then follow up with "How does it handle memory?", the word **"it"** carries the entire meaning of your follow-up — but the retriever would embed the literal text `"How does it handle memory?"` and search for chunks about a mysterious, undefined "it." The vector search has zero conversational context; it only sees whatever string you hand it.

**The fix:** before retrieving anything, we first ask the LLM to **rewrite the follow-up into a standalone question** using the chat history — e.g. turn `"How does it handle memory?"` into `"How does LangChain handle memory?"` — and only *then* run retrieval on that rewritten, self-contained question. This two-step pattern (condense → retrieve → answer) is the standard way conversational RAG is built.

## Step 2 — Install Dependencies (same as Day 4)

In [1]:
!pip install -q langchain langchain-google-genai langchain-community langchain-text-splitters langchain-chroma chromadb pypdf python-dotenv


[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## Step 3 — Setup, Reload Vector Store, Init Model

In [2]:
from dotenv import load_dotenv
import os

load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
assert GEMINI_API_KEY, "GEMINI_API_KEY not found — check your .env file"

from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage

PERSIST_DIR = "chroma_db"

embeddings_model = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=os.getenv("GEMINI_API_KEY")
)
vector_store = Chroma(persist_directory=PERSIST_DIR, embedding_function=embeddings_model)

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=os.getenv("GEMINI_API_KEY")
)

print(f"Reloaded vector store with {vector_store._collection.count()} chunks")

Reloaded vector store with 13 chunks


## Step 4 — Bring Over Day 4's Filtered Retrieval + Citation Logic

Unchanged from yesterday — we're building on top of it, not replacing it.

In [3]:
SCORE_THRESHOLD = 0.8  # keep whatever value you tuned to on Day 4

def retrieve_filtered(query: str, k: int = 5, threshold: float = SCORE_THRESHOLD):
    results = vector_store.similarity_search_with_score(query, k=k)
    return [(doc, score) for doc, score in results if score <= threshold]

def format_docs_with_citations(filtered_results):
    if not filtered_results:
        return "NO_RELEVANT_CONTEXT", {}
    context_parts = []
    citation_map = {}
    for i, (doc, score) in enumerate(filtered_results, 1):
        tag = f"Source {i}"
        source = doc.metadata.get("source", "unknown")
        page = doc.metadata.get("page", "?")
        citation_map[tag] = f"{source} (page {page})"
        context_parts.append(f"[{tag}]\n{doc.page_content}")
    return "\n\n".join(context_parts), citation_map

## Step 5 — The Question Condenser

Given the chat history and a new (possibly ambiguous) question, ask the LLM to rewrite it into a standalone question. Notice this is a **separate, smaller LLM call** from the final answer generation — it has one narrow job and nothing else.

In [4]:
CONDENSE_PROMPT = """Given the conversation history and a follow-up question, rewrite the follow-up question to be a standalone question that includes all necessary context from the history. If the follow-up question is already standalone, return it unchanged. Do not answer the question — only rewrite it.

Conversation history:
{chat_history}

Follow-up question: {question}

Standalone question:"""

condense_prompt = ChatPromptTemplate.from_template(CONDENSE_PROMPT)

def format_history(chat_history):
    if not chat_history:
        return "(no previous conversation)"
    lines = []
    for msg in chat_history:
        role = "User" if isinstance(msg, HumanMessage) else "Assistant"
        lines.append(f"{role}: {msg.content}")
    return "\n".join(lines)

def condense_question(question: str, chat_history: list) -> str:
    if not chat_history:
        return question  # no history yet, nothing to condense
    standalone = (condense_prompt | llm | StrOutputParser()).invoke({
        "chat_history": format_history(chat_history),
        "question": question
    })
    return standalone.strip()

## Step 6 — Updated Answer Prompt (Aware It's Mid-Conversation)

Same grounding + citation rules as Day 4, with one addition: the model is told it's in an ongoing conversation, so its tone can naturally reference the flow (e.g. "Building on that...") without breaking the "only use the provided context" rule.

In [5]:
RAG_PROMPT_V3 = """You are a helpful assistant having an ongoing conversation, answering questions using ONLY the context provided below.

Rules:
- Base your answer strictly on the context. Do not use outside knowledge.
- Reference the relevant [Source N] tag(s) inline when you use information from them.
- If the context is exactly "NO_RELEVANT_CONTEXT", respond only with: "I don't have relevant information in the provided documents to answer that."
- You may refer naturally to earlier parts of the conversation, but never invent facts that aren't in the context.
- Be concise and direct.

Context:
{context}

Question:
{question}

Answer:"""

answer_prompt = ChatPromptTemplate.from_template(RAG_PROMPT_V3)

## Step 7 — The Full Conversational RAG Function

Putting it together: condense → retrieve (on the condensed question) → format → answer → **update memory** with both the original question and the answer.

Note we store the *original* user question in history (not the condensed one) — that keeps the transcript reading naturally, the way the conversation actually happened.

In [6]:
chat_history = []  # persists across calls in this notebook session

def chat(question: str):
    standalone_question = condense_question(question, chat_history)

    if standalone_question != question:
        print(f"(condensed to: \"{standalone_question}\")")

    filtered = retrieve_filtered(standalone_question)
    context, citation_map = format_docs_with_citations(filtered)

    answer = (answer_prompt | llm | StrOutputParser()).invoke({
        "context": context,
        "question": standalone_question
    })

    chat_history.append(HumanMessage(content=question))
    chat_history.append(AIMessage(content=answer))

    print(f"\nQ: {question}")
    print(f"A: {answer}")
    if citation_map:
        print("\nSources:")
        for tag, label in citation_map.items():
            print(f"  {tag} -> {label}")
    print("-" * 60)

    return answer

## Step 8 — Test a Real Multi-Turn Conversation

Ask an initial question, then a deliberately ambiguous follow-up that only makes sense given the first answer.

In [7]:
chat("What is the main topic of this document?")


Q: What is the main topic of this document?
A: The main topics covered in the provided documents are Cloud Computing and Modern Software Development [Source 1, Source 3], Artificial Intelligence and its Impact on Modern Society [Source 2, Source 5], and Data Structures and Algorithms [Source 3, Source 4].

Sources:
  Source 1 -> data\Untitled document.pdf (page 1)
  Source 2 -> data\Untitled document.pdf (page 0)
  Source 3 -> data\Untitled document.pdf (page 1)
  Source 4 -> data\Untitled document.pdf (page 2)
  Source 5 -> data\Untitled document.pdf (page 0)
------------------------------------------------------------


'The main topics covered in the provided documents are Cloud Computing and Modern Software Development [Source 1, Source 3], Artificial Intelligence and its Impact on Modern Society [Source 2, Source 5], and Data Structures and Algorithms [Source 3, Source 4].'

In [8]:
# This follow-up would fail badly without condensing — "it" has no meaning on its own
chat("Can you go into more detail about that?")

(condensed to: "Can you go into more detail about the main topics: Cloud Computing and Modern Software Development, Artificial Intelligence and its Impact on Modern Society, and Data Structures and Algorithms?")

Q: Can you go into more detail about that?
A: Here is more detail on the main topics:

**Cloud Computing and Modern Software Development:**
Cloud computing has transformed how software applications are developed, deployed, and maintained, allowing organizations to rent computing resources on demand instead of purchasing expensive physical servers. This approach reduces infrastructure costs and provides scalability [Source 2]. There are three primary cloud service models: Infrastructure as a Service (IaaS), which provides virtual machines, storage, and networking resources; Platform as a Service (PaaS), which offers development platforms for building and deploying applications without managing servers; and Software as a Service (SaaS), which delivers complete applications over 

"Here is more detail on the main topics:\n\n**Cloud Computing and Modern Software Development:**\nCloud computing has transformed how software applications are developed, deployed, and maintained, allowing organizations to rent computing resources on demand instead of purchasing expensive physical servers. This approach reduces infrastructure costs and provides scalability [Source 2]. There are three primary cloud service models: Infrastructure as a Service (IaaS), which provides virtual machines, storage, and networking resources; Platform as a Service (PaaS), which offers development platforms for building and deploying applications without managing servers; and Software as a Service (SaaS), which delivers complete applications over the internet for users to access through a web browser [Source 2, Source 5].\n\nMicroservices architecture has become popular with cloud computing, where developers create multiple small services that communicate through APIs, performing specific function

In [9]:
# Another natural follow-up, testing pronoun resolution across two turns of history
chat("Does it mention anything related to that elsewhere in the document?")

(condensed to: "Does the document mention anything related to Cloud Computing and Modern Software Development, Artificial Intelligence and its Impact on Modern Society, or Data Structures and Algorithms elsewhere?")

Q: Does it mention anything related to that elsewhere in the document?
A: Yes, the document mentions all three topics:
*   **Cloud Computing and Modern Software Development** is discussed, including how it has transformed software development, the concept of renting computing resources, and the three primary cloud service models: IaaS, PaaS, and SaaS [Source 1]. It also touches on serverless computing and the skills developers need for cloud-native technologies [Source 2].
*   **Artificial Intelligence and its Impact on Modern Society** is covered, defining AI, its influence, and reasons for its adoption, such as increased computational power and large datasets. It also highlights the performance of machine learning algorithms, especially deep learning models, in tasks lik

'Yes, the document mentions all three topics:\n*   **Cloud Computing and Modern Software Development** is discussed, including how it has transformed software development, the concept of renting computing resources, and the three primary cloud service models: IaaS, PaaS, and SaaS [Source 1]. It also touches on serverless computing and the skills developers need for cloud-native technologies [Source 2].\n*   **Artificial Intelligence and its Impact on Modern Society** is covered, defining AI, its influence, and reasons for its adoption, such as increased computational power and large datasets. It also highlights the performance of machine learning algorithms, especially deep learning models, in tasks like image classification and speech recognition [Source 4].\n*   **Data Structures and Algorithms** (DSA) is presented as the foundation of efficient software development, crucial for problems involving searching, sorting, memory management, and optimization [Source 2, Source 5]. Specific 

## Step 9 — Inspect the Full Memory State

Worth looking at directly — this `chat_history` list is the entire "memory" of the conversation. Nothing is remembered beyond what's explicitly stored here (same core lesson as your Day 1 email agent's `messages` list).

In [10]:
for i, msg in enumerate(chat_history):
    role = "User" if isinstance(msg, HumanMessage) else "Assistant"
    print(f"[{i}] {role}: {msg.content[:150]}")

[0] User: What is the main topic of this document?
[1] Assistant: The main topics covered in the provided documents are Cloud Computing and Modern Software Development [Source 1, Source 3], Artificial Intelligence an
[2] User: Can you go into more detail about that?
[3] Assistant: Here is more detail on the main topics:

**Cloud Computing and Modern Software Development:**
Cloud computing has transformed how software application
[4] User: Does it mention anything related to that elsewhere in the document?
[5] Assistant: Yes, the document mentions all three topics:
*   **Cloud Computing and Modern Software Development** is discussed, including how it has transformed so


## Step 10 — Resetting the Conversation

For a real app, you'll want a way to start fresh (e.g. a "New Chat" button later in Day 6's UI). Here, it's just clearing the list.

In [11]:
def reset_conversation():
    chat_history.clear()
    print("Conversation history cleared.")

# reset_conversation()

---
## Day 5 Wrap-Up

Today you fixed the single-turn limitation from Days 3–4:

`New question → condense using chat history → retrieve on standalone question → answer with context → append both turns to memory`

The key insight to remember: **retrieval has no idea what "it" or "that" means on its own** — conversational RAG works by resolving ambiguous follow-ups into standalone questions *before* they ever reach the vector store, not by giving the vector store the history directly.

**What's still missing:** everything is still notebook cells — no shareable interface. **Tomorrow (Day 6)** we wrap all of this (Days 2–5 combined) into a proper Streamlit web app, so this becomes a real, demoable project instead of a sequence of cells.

See `README_Day5.md` for the full write-up of today's concepts.